In [1]:
# Configuration (fill the paths below)
import os
from pathlib import Path
import torch
import numpy as np

# Attribution score and similarity tensors
# score: (N_train, N_test)
# sim:   (L, 2, N_train, N_test) — use layer_index, pooling_index to select
score_pt = "/home/xiruij/anticipation/checkpoints_subset_large/score_LoGra_4096_gen_prompted.pt"
sim_pt    = "/home/xiruij/anticipation/checkpoints_subset_large/audio_similarity_all_layers_gen_prompted.pt"

# Selection hyperparameters
layer_index = 25     # 1-based layer id
pooling_index = 0    # 0 = mean, 1 = max

test_sample_index = 123   # chosen generated sample index in [0, N_test)
top_k = 10              # number of HA-LS training examples to report

# Optional: map train/test indices to filenames (not required for printing indices)
train_dir = "/home/xiruij/anticipation/datasets/finetune/song_train_mp3"  # set to directory with .mp3 if you want filenames printed

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda


In [2]:
# Load score and similarity
score = torch.load(score_pt, map_location="cpu")
sim = torch.load(sim_pt, map_location="cpu")

assert torch.is_tensor(score) and score.ndim == 2, f"score must be (N_train,N_test), got {tuple(score.shape)}"
assert torch.is_tensor(sim) and sim.ndim == 4, f"sim must be (L,2,N_train,N_test), got {tuple(sim.shape)}"

L, P, N_train, N_test = sim.shape
layer_idx0 = max(0, min(L - 1, layer_index - 1))
assert 0 <= pooling_index < P

sim_selected = sim[layer_idx0, pooling_index]  # (N_train, N_test)
assert sim_selected.shape == score.shape

print("Loaded score shape:", tuple(score.shape))
print("Loaded sim_selected shape:", tuple(sim_selected.shape))
print(f"Using layer={layer_index} (0-based {layer_idx0}), pooling={pooling_index}")


/tmp/ipykernel_784695/3031116324.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  score = torch.load(score_pt, map_location="cpu")
/tmp/ipykernel_784695/3031116324.py:3: 

Loaded score shape: (28000, 500)
Loaded sim_selected shape: (28000, 500)
Using layer=25 (0-based 24), pooling=0


In [3]:
# Identify HA-LS (high attribution, low similarity) for the chosen test sample
from dataclasses import dataclass
from typing import List

@dataclass
class PairItem:
    train_idx: int
    rank: int
    score: float
    sim: float
    delta: float

def find_hals_for_test_col(score_m: torch.Tensor,
                           sim_m: torch.Tensor,
                           test_idx: int,
                           top_k: int = 10,
                           alpha_high: float = 0.9,
                           beta_low: float = 0.1) -> List[PairItem]:
    s_col = score_m[:, test_idx]
    m_col = sim_m[:, test_idx]

    qS_high = torch.quantile(s_col, alpha_high).item()
    qM_low  = torch.quantile(m_col, beta_low).item()

    mu_s, std_s = s_col.mean().item(), (s_col.std(unbiased=False).item() or 1e-6)
    mu_m, std_m = m_col.mean().item(), (m_col.std(unbiased=False).item() or 1e-6)
    Zs = (s_col - mu_s) / std_s
    Zm = (m_col - mu_m) / std_m
    delta = (Zm - Zs).numpy()  # negative = HA-LS

    idx_desc = torch.argsort(s_col, descending=True)
    rank = torch.empty_like(idx_desc, dtype=torch.int32)
    rank[idx_desc] = torch.arange(1, idx_desc.size(0) + 1, dtype=torch.int32)

    mask_hals = (s_col >= qS_high) & (m_col <= qM_low)
    cand = torch.nonzero(mask_hals, as_tuple=False).flatten().tolist()
    cand = sorted(cand, key=lambda i: float(delta[i]))  # most negative first

    items: List[PairItem] = []
    for i in cand[:top_k]:
        items.append(PairItem(
            train_idx=int(i),
            rank=int(rank[i].item()),
            score=float(s_col[i].item()),
            sim=float(m_col[i].item()),
            delta=float(delta[i]),
        ))
    return items

hals_items = find_hals_for_test_col(score, sim_selected, test_sample_index, top_k=top_k)
print(f"Top-{top_k} HA-LS training indices for test index {test_sample_index}:")
print([it.train_idx for it in hals_items])

# Optional: print minimal details
for it in hals_items:
    print(f"idx={it.train_idx} rank={it.rank} score={it.score:.6f} sim={it.sim:.6f} delta={it.delta:.6f}")


Top-10 HA-LS training indices for test index 123:
[6830, 10438, 6889, 3083, 14222, 3622, 21234, 21928, 6915, 9683]
idx=6830 rank=577 score=1532.047119 sim=0.629655 delta=-4.036579
idx=10438 rank=108 score=3588.057373 sim=0.702531 delta=-4.025052
idx=6889 rank=87 score=4078.020752 sim=0.721318 delta=-3.999699
idx=3083 rank=321 score=2046.809448 sim=0.665585 delta=-3.752145
idx=14222 rank=126 score=3166.240479 sim=0.714100 delta=-3.605190
idx=3622 rank=227 score=2384.918701 sim=0.690279 delta=-3.547898
idx=21234 rank=209 score=2496.936035 sim=0.697817 delta=-3.490479
idx=21928 rank=232 score=2358.844482 sim=0.693508 delta=-3.481923
idx=6915 rank=159 score=2866.557861 sim=0.714016 delta=-3.439081
idx=9683 rank=587 score=1517.254150 sim=0.669040 delta=-3.401267


In [4]:
# Generation probability comparison for the chosen sample
from transformers import AutoModelForCausalLM
from anticipation.vocab import AUTOREGRESS

# Model paths (fill in)
full_model_path = "/home/xiruij/anticipation/checkpoints_subset_large/removed_random_model_full_2"
removed_topk_model_path = "/home/xiruij/anticipation/checkpoints_subset_large/removed_hals_model_1"
removed_random_model_path = "/home/xiruij/anticipation/checkpoints_subset_large/removed_random_model_3"

generated_samples_file = "/home/xiruij/anticipation/datasets/finetune/generated_samples_prompted.txt"
generated_sample_index = test_sample_index  # the same sample you care about

print("Using generated sample index:", generated_sample_index)


def load_model(model_path: str, device: torch.device):
    model = AutoModelForCausalLM.from_pretrained(model_path, attn_implementation="eager").to(device)
    model.eval()
    return model


def load_generated_batch(file_path: str, sample_index: int, device: torch.device, max_length: int | None = None):
    with open(file_path, "r", encoding="utf-8") as f:
        lines = f.readlines()
    assert 0 <= sample_index < len(lines), f"sample_index={sample_index} out of range (len={len(lines)})"
    arr = np.fromstring(lines[sample_index].strip(), dtype=int, sep=" ")
    if arr.size == 0:
        input_ids = np.array([AUTOREGRESS], dtype=int)
    else:
        input_ids = np.concatenate([np.array([AUTOREGRESS], dtype=int), arr])
    if max_length is not None:
        input_ids = input_ids[:max_length]
    input_ids_t = torch.tensor(input_ids, dtype=torch.long).unsqueeze(0)
    batch = {
        "input_ids": input_ids_t.to(device),
        "attention_mask": torch.ones_like(input_ids_t, dtype=torch.long).to(device),
        "labels": input_ids_t.to(device),
    }
    return batch, int(input_ids_t.size(1))


def compute_metrics(model, batch: dict, num_tokens: int):
    with torch.no_grad():
        outputs = model(batch["input_ids"], attention_mask=batch["attention_mask"], labels=batch["labels"])  # mean NLL over tokens
        nll_mean = float(outputs.loss.item())
    return {
        "n_tokens": num_tokens,
        "nll_mean": nll_mean,
        "avg_log_prob": -nll_mean,
        "sum_log_prob": -nll_mean * num_tokens,
        "ppl": float(np.exp(nll_mean)),
    }

models = {
    "full": full_model_path,
    "removed_topk": removed_topk_model_path,
    "removed_random": removed_random_model_path,
}

results = {}
for name, mpath in models.items():
    if not mpath or not os.path.isdir(mpath):
        print(f"Skip {name}: invalid path -> {mpath}")
        continue
    print(f"Loading {name} from {mpath}")
    model = load_model(mpath, device)
    batch, n_tokens = load_generated_batch(generated_samples_file, generated_sample_index, device)
    metrics = compute_metrics(model, batch, n_tokens)
    results[name] = metrics
    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()

print("\nGeneration probability metrics (mean NLL / log-prob):")
for name, m in results.items():
    print(f"{name:>14} | n_tokens={m['n_tokens']:4d} | avg_log_prob={m['avg_log_prob']:.6f} | sum_log_prob={m['sum_log_prob']:.3f} | ppl={m['ppl']:.4f}")


/home/xiruij/miniconda3/envs/dattri/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using generated sample index: 123
Loading full from /home/xiruij/anticipation/checkpoints_subset_large/removed_random_model_full_2


`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Loading removed_topk from /home/xiruij/anticipation/checkpoints_subset_large/removed_hals_model_1
Loading removed_random from /home/xiruij/anticipation/checkpoints_subset_large/removed_random_model_3

Generation probability metrics (mean NLL / log-prob):
          full | n_tokens=1024 | avg_log_prob=-1.226742 | sum_log_prob=-1256.184 | ppl=3.4101
  removed_topk | n_tokens=1024 | avg_log_prob=-1.232594 | sum_log_prob=-1262.177 | ppl=3.4301
removed_random | n_tokens=1024 | avg_log_prob=-1.105086 | sum_log_prob=-1131.608 | ppl=3.0195


In [5]:
# Test cell - verify that the code works with actual model and data
test_model_path = "/home/xiruij/anticipation/checkpoints_subset_large/full_model"
test_samples_file = "/home/xiruij/anticipation/datasets/finetune/generated_samples_prompted.txt"
test_sample_idx = 0

print("Loading test model...")
test_model = load_model(test_model_path, device)
print("Model loaded successfully!")

print(f"\nLoading sample {test_sample_idx} from {test_samples_file}...")
test_batch, test_n_tokens = load_generated_batch(test_samples_file, test_sample_idx, device)
print(f"Sample loaded: {test_n_tokens} tokens")

print("\nComputing metrics...")
test_metrics = compute_metrics(test_model, test_batch, test_n_tokens)

print("\nTest results:")
print(f"  n_tokens      = {test_metrics['n_tokens']}")
print(f"  nll_mean      = {test_metrics['nll_mean']:.6f}")
print(f"  avg_log_prob  = {test_metrics['avg_log_prob']:.6f}")
print(f"  sum_log_prob  = {test_metrics['sum_log_prob']:.3f}")
print(f"  ppl           = {test_metrics['ppl']:.4f}")

print("\n✓ Test passed! The code works correctly.")

del test_model
if device.type == "cuda":
    torch.cuda.empty_cache()


Loading test model...
Model loaded successfully!

Loading sample 0 from /home/xiruij/anticipation/datasets/finetune/generated_samples_prompted.txt...
Sample loaded: 1024 tokens

Computing metrics...

Test results:
  n_tokens      = 1024
  nll_mean      = 1.041066
  avg_log_prob  = -1.041066
  sum_log_prob  = -1066.051
  ppl           = 2.8322

✓ Test passed! The code works correctly.


In [6]:
# Check random indices for comparison
print("="*80)
print("Random indices check for test sample", test_sample_index)
print("="*80)

random_indices = [17896, 1823, 2186, 8875, 7788, 7466, 25088, 7135, 2708, 18689]
s_col = score[:, test_sample_index]
m_col = sim_selected[:, test_sample_index]

# Compute ranks
idx_desc = torch.argsort(s_col, descending=True)
rank_mat = torch.empty_like(idx_desc, dtype=torch.int32)
rank_mat[idx_desc] = torch.arange(1, idx_desc.size(0) + 1, dtype=torch.int32)

print(f"\n{'idx':>6} | {'rank':>6} | {'score':>12} | {'sim':>8}")
print("-" * 45)
for idx in random_indices:
    if idx < len(s_col):
        print(f"{idx:6d} | {rank_mat[idx].item():6d} | {s_col[idx].item():12.6f} | {m_col[idx].item():8.6f}")
    else:
        print(f"{idx:6d} | OUT OF RANGE (max={len(s_col)-1})")


Random indices check for test sample 123

   idx |   rank |        score |      sim
---------------------------------------------
 17896 |   4701 |   481.251953 | 0.874285
  1823 |  15276 |   -78.354767 | 0.866199
  2186 |  10525 |   136.757446 | 0.818409
  8875 |   2274 |   792.684448 | 0.822338
  7788 |  14239 |   -33.005203 | 0.886040
  7466 |  17974 |  -199.163849 | 0.730449
 25088 |  11113 |   109.542816 | 0.895564
  7135 |  14569 |   -46.318241 | 0.866034
  2708 |  13067 |    19.579857 | 0.914699
 18689 |  26505 |  -920.267090 | 0.803678
